# Notebook 6: Experiment 2 — Cross-Stock Prediction (70/30)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on one stock's daily data, predict on another stock's daily test data.  
**Train/Test Split:** 70/30 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501 BBCA ATH)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.7
RATIO_LABEL = '70_30'
EXP_LABEL = f'Exp2_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 2 - Cross-Stock Prediction (70/30)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 2 - Cross-Stock Prediction (70/30)


In [ ]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


In [ ]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

## Run All Cross-Stock Experiments

### Methodology: Zero-Shot Transfer Learning
Instead of retraining from scratch, we load pre-trained models from **Experiment 1** (with same-stock data) and directly apply them to predict different stocks' test data. This tests how well models generalize across stocks without any domain adaptation.

**Expected result:** Performance will likely be worse than same-stock predictions due to different price ranges, volatility, and patterns across stocks. However, this shows raw transfer capability.

In [ ]:
# ============================================================
# EXPERIMENT 2: Cross-stock prediction (using Exp1 pre-trained models)
# Load models trained on Stock A, test on Stock B (zero-shot transfer)
# ============================================================
from tensorflow.keras.models import load_model

all_results = []
all_predictions = {}  # {(train_stock, test_stock): {model_type: (y_true, y_pred, dates)}}

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue  # Skip same-stock (covered in Exp 1)
        
        pair_key = (train_stock, test_stock)
        print(f"\n{'#'*60}")
        print(f"# TRAIN: {train_stock} -> TEST: {test_stock}")
        print(f"# Using pre-trained Exp1 models (zero-shot transfer)")
        print(f"{'#'*60}")
        
        # Prepare cross-stock data
        X_train, y_train, X_test, y_test, test_dates = prepare_cross_stock_data(
            daily_data[train_stock], daily_data[test_stock],
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        all_predictions[pair_key] = {}
        
        for model_type in MODEL_TYPES:
            # Build Exp1 model filename
            exp1_model_path = f'models/Exp1_70_30/Exp1_70_30_{train_stock}_{model_type}_best.keras'
            
            if not os.path.exists(exp1_model_path):
                print(f"  ⚠️  Model not found: {exp1_model_path}")
                continue
            
            print(f"\n  Loading {model_type} from: {exp1_model_path}")
            
            # Load pre-trained model
            model = load_model(exp1_model_path)
            
            # Make predictions on test data (NO training)
            y_pred_scaled = model.predict(X_test, verbose=0).flatten()
            
            # Inverse scale to original values
            y_true_inv = proportion_inverse_scale(y_test)
            y_pred_inv = proportion_inverse_scale(y_pred_scaled)
            
            # Evaluate
            metrics = evaluate_predictions(y_true_inv, y_pred_inv)
            
            result = {
                'Train_Stock': train_stock,
                'Test_Stock': test_stock,
                'Model': model_type,
                **metrics
            }
            all_results.append(result)
            all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
            
            print(f"    RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}, R²: {metrics['R2']:.6f}")
            
            # Plot prediction
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'Train_{train_stock}_Test_{test_stock}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 2 (70/30) cross-stock prediction complete!")


############################################################
# TRAIN: TLKM -> TEST: BBCA
############################################################
  X_train: (3610, 60, 1), X_test: (1574, 60, 1)

Training BiLSTM for: Exp2_70_30_train_TLKM_test_BBCA
  Train samples: 3610, Test samples: 1574
Epoch 1/100
50/51 [============================>.] - ETA: 0s - loss: 0.0011
Epoch 1: val_loss improved from inf to 0.00017, saving model to models/Exp2_70_30\Exp2_70_30_train_TLKM_test_BBCA_BiLSTM_best.keras
51/51 [==============================] - 18s 96ms/step - loss: 0.0010 - val_loss: 1.6579e-04
Epoch 2/100
51/51 [==============================] - ETA: 0s - loss: 1.1912e-04
Epoch 2: val_loss improved from 0.00017 to 0.00013, saving model to models/Exp2_70_30\Exp2_70_30_train_TLKM_test_BBCA_BiLSTM_best.keras
51/51 [==============================] - 2s 47ms/step - loss: 1.1912e-04 - val_loss: 1.3167e-04
Epoch 3/100
51/51 [==============================] - ETA: 0s - loss: 9.8347e-05
Epoch 3: val

## Results Summary

In [ ]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 2 - Cross-Stock Prediction (70/30)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 2 - Cross-Stock Prediction (70/30)
Train_Stock Test_Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
       TLKM       BBCA BiLSTM 216669.0033 465.4772 426.5887    5.6963 0.913256            184.0         100
       TLKM       BBCA  BiGRU  37057.0097 192.5020 157.8625    2.1559 0.985164            164.1         100
       TLKM       BBCA   LSTM  78862.7241 280.8251 233.3805    3.1024 0.968427             99.4         100
       TLKM       BBCA    GRU  65386.7279 255.7083 218.5628    2.9415 0.973822             95.3         100
       TLKM       ASII BiLSTM  72151.0834 268.6095 237.5453    5.2150 0.864407            167.8         100
       TLKM       ASII  BiGRU  19611.5673 140.0413 106.5659    2.4091 0.963144            168.0         100
       TLKM       ASII   LSTM  21902.9293 147.9964 115.2728    2.6127 0.958838             99.3         100
       TLKM       ASII    GRU  21271.4368 145.8473 117.5976    2.6369 0.960025         

## Visualizations

In [ ]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS - CROSS-STOCK
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Cross-Stock Results Dashboard
print("1. Generating Cross-Stock Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp2(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Transfer Learning Matrix Heatmaps
print("2. Generating Transfer Learning Matrices...")
for metric in ['RMSE', 'MAE', 'R2']:
    try:
        create_interactive_transfer_matrix(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"   ✓ {metric} transfer matrices generated")
    except Exception as e:
        print(f"   ⚠ Skipping {metric}: {str(e)}")

# Display one example
fig_example, html_example = create_interactive_transfer_matrix(
    results_df, 'RMSE', EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print("   (Displaying BiLSTM transfer matrix example)\n")

print()

# 3. Metrics Comparison Chart
print("3. Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("4. Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html4}")
fig4.show()

print("\n✓ All interactive results visualizations generated successfully!")

print("\n" + "="*70)

### Interactive Results Visualizations

In [ ]:
# ============================================================
# HEATMAPS PER MODEL
# ============================================================
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_heatmap(
        results_df, metric, EXP_LABEL,
        row_col='Train_Stock', col_col='Test_Stock',
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All heatmaps saved!")


In [ ]:
# ============================================================
# COMPARISON: All models for each train->test pair
# ============================================================
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        preds = {mt: all_predictions[pair_key][mt][1] for mt in MODEL_TYPES if mt in all_predictions[pair_key]}
        
        plot_all_models_comparison(
            dates, y_true, preds,
            f'Train_{train_stock}_Test_{test_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("All comparison plots saved!")


In [ ]:
# ============================================================
# SUMMARY: BEST MODEL PER CROSS-STOCK PAIR
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER PAIR (by RMSE)")
print("="*70)
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_data = results_df[
            (results_df['Train_Stock'] == train_stock) &
            (results_df['Test_Stock'] == test_stock)
        ]
        if pair_data.empty:
            continue
        best_idx = pair_data['RMSE'].idxmin()
        best = pair_data.loc[best_idx]
        print(f"  {train_stock} -> {test_stock}: {best['Model']} "
              f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")


## Interactive Visualization (Plotly)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ALL MODELS COMPARISON (Plotly)
# Actual Prices in RED (#FF0000), Models in Model Colors
# ============================================================
print("Generating interactive plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating interactive plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with all models visible
        fig = go.Figure()
        
        # Linestyles for variety
        linestyles_map = {
            'BiLSTM': 'solid',
            'BiGRU': 'dash',
            'LSTM': 'dot',
            'GRU': 'dashdot'
        }
        
        # Add actual values in RED (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=3, dash='solid'),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True,
            opacity=0.95
        ))
        
        # Add all model predictions
        for model_type in MODEL_TYPES:
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=f'{model_type} (Predicted)',
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2.5,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                    visible=True,
                    opacity=0.85
                ))
        
        # Update layout
        fig.update_layout(
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>Zero-Shot Transfer Learning (70/30 Split)</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=20, family="Times New Roman"),
            height=700,
            width=1200,
            margin=dict(l=80, r=80, t=120, b=80),
            xaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=16, family="Times New Roman")
            ),
            yaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=16, family="Times New Roman")
            ),
            legend=dict(
                x=0.01,
                y=0.99,
                xanchor="left",
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="gray",
                borderwidth=1,
                font=dict(size=18, family="Times New Roman")
            )
        )
        
        # Add range slider
        fig.update_xaxes(rangeslider_visible=False)
        
        # Save ALL MODELS plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_all_models.html'
        fig.write_html(html_filename)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")
print(f"  Actual prices displayed in RED (#FF0000)")
print(f"  Font: Times New Roman, Size: 20")

In [ ]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE BUTTONS (Plotly)
# Actual Prices in RED (#FF0000)
# ============================================================
print("Generating interactive toggle plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating toggle plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with toggle buttons
        fig = go.Figure()
        
        linestyles_map = {
            'BiLSTM': 'solid',
            'BiGRU': 'dash',
            'LSTM': 'dot',
            'GRU': 'dashdot'
        }
        
        # Add actual values in RED (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=3, dash='solid'),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True,
            opacity=0.95
        ))
        
        # Add each model with visibility control
        for idx, model_type in enumerate(MODEL_TYPES):
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                # First model visible by default, others hidden
                is_visible = True if idx == 0 else False
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=f'{model_type} (Predicted)',
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2.5,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                    visible=is_visible,
                    opacity=0.85
                ))
        
        # Create buttons for toggling models
        buttons = []
        for i, model_type in enumerate(MODEL_TYPES):
            # Create visibility list: [True for Actual, False for all models except this one, True for this model]
            visibility = [True] + [j == i for j in range(len(MODEL_TYPES))]
            
            buttons.append(
                dict(
                    label=model_type,
                    method='update',
                    args=[
                        {'visible': visibility},
                        {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>{model_type} Model (70/30 Split)</sub>'}
                    ]
                )
            )
        
        # Add "Show All" button
        buttons.insert(0, dict(
            label='All Models',
            method='update',
            args=[
                {'visible': [True] * (len(MODEL_TYPES) + 1)},
                {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison (70/30 Split)</sub>'}
            ]
        ))
        
        # Update layout with buttons
        fig.update_layout(
            updatemenus=[
                dict(
                    active=0,
                    buttons=buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=0.99,
                    yanchor="top",
                    bgcolor="rgba(200, 200, 200, 0.9)",
                    bordercolor="gray",
                    borderwidth=2,
                    font=dict(size=16, family="Times New Roman")
                )
            ],
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison (70/30 Split)</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=20, family="Times New Roman"),
            height=700,
            width=1200,
            margin=dict(l=80, r=80, t=150, b=80),
            xaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=16, family="Times New Roman")
            ),
            yaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=16, family="Times New Roman")
            ),
            legend=dict(
                x=0.99,
                y=0.01,
                xanchor="right",
                yanchor="bottom",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="gray",
                borderwidth=1,
                font=dict(size=18, family="Times New Roman")
            )
        )
        
        # Save TOGGLE plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_toggle.html'
        fig.write_html(html_filename)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive toggle plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*_toggle.html")
print(f"  Actual prices displayed in RED (#FF0000)")
print(f"  Font: Times New Roman, Size: 20")

## Case-by-Case Interactive Actual vs Predicted
One interactive Plotly chart per notebook with a dropdown that walks through every test case. Each case shows the Actual price (red) plus all four model predictions, with a metrics panel (MSE, RMSE, MAE, MAPE, R²) for that case.

In [ ]:
# ============================================================
# CASE-BY-CASE INTERACTIVE ACTUAL VS PREDICTED (Experiment 2)
# ============================================================
# Cases follow the test plan: 12 train -> test cross-stock pairs.
from collections import OrderedDict

cases_dict = OrderedDict()
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        available_mts = [mt for mt in MODEL_TYPES if mt in all_predictions[pair_key]]
        if not available_mts:
            continue
        y_true, _, dates = all_predictions[pair_key][available_mts[0]]
        predictions = {mt: all_predictions[pair_key][mt][1] for mt in available_mts}

        metrics = {}
        for mt in MODEL_TYPES:
            row = results_df[(results_df['Train_Stock'] == train_stock) &
                             (results_df['Test_Stock']  == test_stock) &
                             (results_df['Model']       == mt)]
            if not row.empty:
                r = row.iloc[0]
                metrics[mt] = {
                    'MSE'      : r.get('MSE'),
                    'RMSE'     : r.get('RMSE'),
                    'MAE'      : r.get('MAE'),
                    'MAPE (%)' : r.get('MAPE (%)'),
                    'R2'       : r.get('R2'),
                }

        label = f'{train_stock}→{test_stock}'
        cases_dict[label] = {
            'description' : f'Train {train_stock} Daily → Predict {test_stock} Daily',
            'dates'       : dates,
            'y_true'      : y_true,
            'predictions' : predictions,
            'metrics'     : metrics,
        }

ratio_pretty = RATIO_LABEL.replace('_', '/')
fig_case, html_case = create_case_by_case_actual_vs_predicted(
    cases_dict,
    experiment_title=f'Experiment 2: Cross-Stock Prediction ({ratio_pretty})',
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}',
)
print(f"✓ Case-by-case visualization saved: {html_case}")
print(f"  Cases ({len(cases_dict)}): {list(cases_dict.keys())}")
fig_case.show()
